# Chapter 17 Lab: Experiments, Error Analysis, and Scientific Claims (`ch10`)

This notebook explores the core concepts of experimental design in machine learning:
- Why a single run is unreliable (repeated runs with randomness)
- Ablation studies to identify feature importance
- Slice analysis to reveal subgroup performance disparities
- Simpson's paradox and aggregate vs. slice metrics
- Making honest claims about model performance

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## 1. Repeated Runs: Why a Single Run Is Unreliable

Train a logistic regression model on `make_moons` data 20 times with different random seeds. Each run creates a different train/test split and trains from scratch. Record test accuracy for each run.

In [ ]:
np.random.seed(42)
n_runs = 20
accuracies = []

# Generate base dataset (same data, different splits each time)
X, y = make_moons(n_samples=400, noise=0.15, random_state=42)

for seed in range(n_runs):
    # Different train/test split each time
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=seed
    )
    
    # Scale and train
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = LogisticRegression(random_state=seed, max_iter=1000)
    model.fit(X_train_scaled, y_train)
    
    acc = model.score(X_test_scaled, y_test)
    accuracies.append(acc)

accuracies = np.array(accuracies)
mean_acc = accuracies.mean()
std_acc = accuracies.std()

print(f"Mean accuracy: {mean_acc:.4f} ± {std_acc:.4f}")
print(f"Min accuracy: {accuracies.min():.4f}")
print(f"Max accuracy: {accuracies.max():.4f}")
print(f"\nIndividual run accuracies:")
for i, acc in enumerate(accuracies):
    print(f"  Run {i+1:2d}: {acc:.4f}")

In [ ]:
# Plot distribution of accuracies across runs
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(accuracies, bins=8, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(mean_acc, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_acc:.4f}')
axes[0].axvline(mean_acc - std_acc, color='orange', linestyle=':', linewidth=2, label=f'±1 std')
axes[0].axvline(mean_acc + std_acc, color='orange', linestyle=':', linewidth=2)
axes[0].set_xlabel('Test Accuracy')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Test Accuracies (20 runs)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Line plot showing variability
axes[1].plot(range(1, n_runs + 1), accuracies, 'o-', color='steelblue', markersize=6)
axes[1].axhline(mean_acc, color='red', linestyle='--', linewidth=2, label=f'Mean')
axes[1].fill_between(
    range(1, n_runs + 1),
    mean_acc - std_acc,
    mean_acc + std_acc,
    alpha=0.2,
    color='red'
)
axes[1].set_xlabel('Run Number')
axes[1].set_ylabel('Test Accuracy')
axes[1].set_title('Test Accuracy Across Different Random Seeds')
axes[1].set_ylim([accuracies.min() - 0.02, accuracies.max() + 0.02])
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nKey insight: A single run could report accuracy anywhere from {accuracies.min():.4f} to {accuracies.max():.4f}.")
print(f"The range is {accuracies.max() - accuracies.min():.4f}, which is substantial!")

## 2. Ablation Study: Feature Importance

Generate synthetic data with 5 features: 3 are useful for prediction, 2 are pure noise. Train a model on all 5 features, then ablate (remove) each feature one at a time and measure the accuracy drop.

In [ ]:
np.random.seed(42)

# Generate data: 3 useful + 2 noise features
n_samples = 500
X_useful, y = make_classification(
    n_samples=n_samples,
    n_features=3,
    n_informative=3,
    n_redundant=0,
    random_state=42
)

# Add 2 pure noise features
X_noise = np.random.randn(n_samples, 2)
X_full = np.hstack([X_useful, X_noise])

feature_names = ['Feature 1 (useful)', 'Feature 2 (useful)', 'Feature 3 (useful)',
                 'Feature 4 (noise)', 'Feature 5 (noise)']

# Train on all features
X_train, X_test, y_train, y_test = train_test_split(X_full, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_full = LogisticRegression(random_state=42, max_iter=1000)
model_full.fit(X_train_scaled, y_train)
baseline_acc = model_full.score(X_test_scaled, y_test)

print(f"Baseline accuracy (all 5 features): {baseline_acc:.4f}")

# Ablation: remove each feature, retrain, measure drop
accuracy_drops = []
feature_indices = list(range(5))

for i, feature_idx in enumerate(feature_indices):
    # Keep all features except this one
    keep_indices = [j for j in feature_indices if j != feature_idx]
    
    X_train_ablated = X_train[:, keep_indices]
    X_test_ablated = X_test[:, keep_indices]
    
    scaler_ablated = StandardScaler()
    X_train_ablated_scaled = scaler_ablated.fit_transform(X_train_ablated)
    X_test_ablated_scaled = scaler_ablated.transform(X_test_ablated)
    
    model_ablated = LogisticRegression(random_state=42, max_iter=1000)
    model_ablated.fit(X_train_ablated_scaled, y_train)
    ablated_acc = model_ablated.score(X_test_ablated_scaled, y_test)
    
    drop = baseline_acc - ablated_acc
    accuracy_drops.append(drop)
    
    print(f"Ablated {feature_names[i]:25s}: accuracy = {ablated_acc:.4f}, drop = {drop:.4f}")

accuracy_drops = np.array(accuracy_drops)

In [ ]:
# Plot ablation results
fig, ax = plt.subplots(figsize=(10, 5))

colors = ['#d62728' if i < 3 else '#1f77b4' for i in range(5)]  # Red for useful, blue for noise
bars = ax.bar(range(5), accuracy_drops, color=colors, edgecolor='black', alpha=0.7)

ax.set_xticks(range(5))
ax.set_xticklabels(feature_names, rotation=45, ha='right')
ax.set_ylabel('Accuracy Drop When Ablated', fontsize=11)
ax.set_title('Ablation Study: Impact of Removing Each Feature', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, (bar, drop) in enumerate(zip(bars, accuracy_drops)):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{drop:.4f}', ha='center', va='bottom', fontsize=10)

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#d62728', alpha=0.7, label='Useful features'),
                   Patch(facecolor='#1f77b4', alpha=0.7, label='Noise features')]
ax.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.show()

print(f"\nKey insight: Useful features have high ablation impact ({accuracy_drops[:3].mean():.4f}).")
print(f"Noise features have minimal impact ({accuracy_drops[3:].mean():.4f}).")

## 3. Slice Analysis / Error Analysis

Train a model on data with two subgroups: Group A (easy to classify) and Group B (hard to classify). Show that overall accuracy looks good, but Group B has much worse performance.

In [ ]:
np.random.seed(42)

# Create two subgroups with different difficulty
# Group A: well-separated, easy to classify
X_A, y_A = make_classification(
    n_samples=200,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    flip_y=0.05,  # 5% label noise
    random_state=42
)

# Group B: overlapped, harder to classify
X_B, y_B = make_classification(
    n_samples=200,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    flip_y=0.25,  # 25% label noise
    random_state=43
)
# Shift group B
X_B = X_B + 3

# Combine
X_combined = np.vstack([X_A, X_B])
y_combined = np.hstack([y_A, y_B])
group_labels = np.hstack([np.zeros(200), np.ones(200)])  # 0=Group A, 1=Group B

# Train/test split
X_train, X_test, y_train, y_test, groups_train, groups_test = train_test_split(
    X_combined, y_combined, group_labels, test_size=0.3, random_state=42
)

# Train model
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

# Overall metrics
y_pred = model.predict(X_test_scaled)
overall_acc = accuracy_score(y_test, y_pred)

# Per-group metrics
mask_A = groups_test == 0
mask_B = groups_test == 1

acc_A = accuracy_score(y_test[mask_A], y_pred[mask_A])
acc_B = accuracy_score(y_test[mask_B], y_pred[mask_B])

prec_A = precision_score(y_test[mask_A], y_pred[mask_A], zero_division=0)
prec_B = precision_score(y_test[mask_B], y_pred[mask_B], zero_division=0)

rec_A = recall_score(y_test[mask_A], y_pred[mask_A], zero_division=0)
rec_B = recall_score(y_test[mask_B], y_pred[mask_B], zero_division=0)

print(f"Overall accuracy: {overall_acc:.4f}")
print(f"\nGroup A (easy):")
print(f"  Accuracy:  {acc_A:.4f}")
print(f"  Precision: {prec_A:.4f}")
print(f"  Recall:    {rec_A:.4f}")
print(f"\nGroup B (hard):")
print(f"  Accuracy:  {acc_B:.4f}")
print(f"  Precision: {prec_B:.4f}")
print(f"  Recall:    {rec_B:.4f}")
print(f"\n*** Gap: Group B accuracy is {(acc_A - acc_B):.4f} worse than Group A ***")

In [ ]:
# Plot confusion matrices side-by-side
cm_A = confusion_matrix(y_test[mask_A], y_pred[mask_A])
cm_B = confusion_matrix(y_test[mask_B], y_pred[mask_B])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Group A
im1 = axes[0].imshow(cm_A, cmap='Blues', aspect='auto')
axes[0].set_title(f'Group A (Easy)\nAccuracy: {acc_A:.4f}', fontweight='bold')
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(['Pred 0', 'Pred 1'])
axes[0].set_yticklabels(['True 0', 'True 1'])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm_A[i, j]), ha='center', va='center',
                     color='white' if cm_A[i, j] > cm_A.max() / 2 else 'black', fontsize=12)
plt.colorbar(im1, ax=axes[0])

# Group B
im2 = axes[1].imshow(cm_B, cmap='Blues', aspect='auto')
axes[1].set_title(f'Group B (Hard)\nAccuracy: {acc_B:.4f}', fontweight='bold')
axes[1].set_xticks([0, 1])
axes[1].set_yticks([0, 1])
axes[1].set_xticklabels(['Pred 0', 'Pred 1'])
axes[1].set_yticklabels(['True 0', 'True 1'])
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, str(cm_B[i, j]), ha='center', va='center',
                     color='white' if cm_B[i, j] > cm_B.max() / 2 else 'black', fontsize=12)
plt.colorbar(im2, ax=axes[1])

# Metric comparison
metrics = ['Accuracy', 'Precision', 'Recall']
group_A_vals = [acc_A, prec_A, rec_A]
group_B_vals = [acc_B, prec_B, rec_B]

x = np.arange(len(metrics))
width = 0.35

axes[2].bar(x - width/2, group_A_vals, width, label='Group A (easy)', color='#2ca02c', alpha=0.7)
axes[2].bar(x + width/2, group_B_vals, width, label='Group B (hard)', color='#d62728', alpha=0.7)
axes[2].set_ylabel('Score')
axes[2].set_title('Per-Group Performance Comparison')
axes[2].set_xticks(x)
axes[2].set_xticklabels(metrics)
axes[2].legend()
axes[2].set_ylim([0, 1.0])
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Simpson's Paradox: Aggregate vs. Slice Metrics

Construct a scenario where Model A beats Model B in overall accuracy, but Model B beats Model A within each subgroup. This reversal is possible because the subgroup composition differs across the two models.

In [ ]:
# Simpson's Paradox Example
# Two models, two subgroups, with different subgroup composition by model

# Group A (easier cases): Model A is evaluated on many Group A cases;
# Model B is evaluated on fewer Group A cases.
group_A_total_model_a = 500
group_A_correct_model_a = 450  # 90% accuracy
group_A_total_model_b = 100
group_A_correct_model_b = 95   # 95% accuracy

# Group B (harder cases): Model A is evaluated on few Group B cases;
# Model B is evaluated on many Group B cases.
group_B_total_model_a = 100
group_B_correct_model_a = 20   # 20% accuracy
group_B_total_model_b = 1000
group_B_correct_model_b = 300  # 30% accuracy

# Compute overall accuracy
total_samples_a = group_A_total_model_a + group_B_total_model_a
total_samples_b = group_A_total_model_b + group_B_total_model_b
model_a_correct = group_A_correct_model_a + group_B_correct_model_a
model_b_correct = group_A_correct_model_b + group_B_correct_model_b

overall_acc_a = model_a_correct / total_samples_a
overall_acc_b = model_b_correct / total_samples_b

acc_a_group_a = group_A_correct_model_a / group_A_total_model_a
acc_a_group_b = group_B_correct_model_a / group_B_total_model_a

acc_b_group_a = group_A_correct_model_b / group_A_total_model_b
acc_b_group_b = group_B_correct_model_b / group_B_total_model_b

print("Simpson's Paradox Example")
print("=" * 60)
print(f"\nOverall Performance:")
print(f"  Model A: {overall_acc_a:.4f} ({model_a_correct}/{total_samples_a})")
print(f"  Model B: {overall_acc_b:.4f} ({model_b_correct}/{total_samples_b})")
print(f"  Winner: Model {'A' if overall_acc_a > overall_acc_b else 'B'} (by overall accuracy)")

print(f"\nGroup A Performance (n_A={group_A_total_model_a}, n_B={group_A_total_model_b}):")
print(f"  Model A: {acc_a_group_a:.4f}")
print(f"  Model B: {acc_b_group_a:.4f}")
print(f"  Better: Model {'A' if acc_a_group_a > acc_b_group_a else 'B'}")

print(f"\nGroup B Performance (n_A={group_B_total_model_a}, n_B={group_B_total_model_b}):")
print(f"  Model A: {acc_a_group_b:.4f}")
print(f"  Model B: {acc_b_group_b:.4f}")
print(f"  Better: Model {'A' if acc_a_group_b > acc_b_group_b else 'B'}")

print(f"\n*** PARADOX: Model A wins overall, but Model B wins in both subgroups! ***")

In [ ]:
# Visualize Simpson's Paradox
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Table view
ax = axes[0]
ax.axis('tight')
ax.axis('off')

table_data = [
    ['', 'Model A', 'Model B'],
    ['Overall', f'{overall_acc_a:.2%}', f'{overall_acc_b:.2%}'],
    ['Group A', f'{acc_a_group_a:.2%}', f'{acc_b_group_a:.2%}'],
    ['Group B', f'{acc_a_group_b:.2%}', f'{acc_b_group_b:.2%}']
]

table = ax.table(cellText=table_data, cellLoc='center', loc='center',
                colWidths=[0.25, 0.35, 0.35])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)

# Style header
for i in range(3):
    table[(0, i)].set_facecolor('#4472C4')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Highlight the paradox
table[(1, 1)].set_facecolor('#92D050')  # Model A wins overall
table[(2, 2)].set_facecolor('#92D050')  # Model B wins in A
table[(3, 2)].set_facecolor('#92D050')  # Model B wins in B

ax.set_title('Simpson\'s Paradox: Overall vs. Subgroup Performance', 
             fontsize=12, fontweight='bold', pad=20)

# Bar chart
ax = axes[1]
x = np.arange(3)
width = 0.35

model_a_accs = [overall_acc_a, acc_a_group_a, acc_a_group_b]
model_b_accs = [overall_acc_b, acc_b_group_a, acc_b_group_b]

bars1 = ax.bar(x - width/2, model_a_accs, width, label='Model A', color='#4472C4', alpha=0.7)
bars2 = ax.bar(x + width/2, model_b_accs, width, label='Model B', color='#ED7D31', alpha=0.7)

ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('Accuracy Comparison: Model A vs. Model B', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(['Overall', 'Group A', 'Group B'])
ax.legend(fontsize=11)
ax.set_ylim([0, 1.0])
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{height:.1%}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 5. Claim Audit: Making Honest Claims

Take the claim: "Our model achieves 95% accuracy." Audit it by asking critical questions: What's the test set? What's the baseline? What's the confidence interval? Are there subgroup failures?

In [ ]:
# Realistic claim scenario
np.random.seed(42)

# Train a model
X, y = make_moons(n_samples=500, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Create synthetic subgroups (e.g., geographic regions, demographic groups)
subgroup = np.random.binomial(1, 0.5, len(X_test))
# Make one subgroup harder
y_test_adj = y_test.copy()
y_test_adj[subgroup == 1] = (y_test_adj[subgroup == 1] + np.random.binomial(1, 0.3, subgroup.sum())) % 2

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

# Compute various metrics
overall_acc = accuracy_score(y_test, y_pred)

# Baseline: always predict majority class
baseline_acc = max(y_test.mean(), 1 - y_test.mean())

# Confidence interval using simple bootstrap
bootstrap_accs = []
for _ in range(1000):
    indices = np.random.choice(len(y_test), len(y_test), replace=True)
    bootstrap_accs.append(accuracy_score(y_test[indices], y_pred[indices]))
bootstrap_accs = np.array(bootstrap_accs)
ci_lower = np.percentile(bootstrap_accs, 2.5)
ci_upper = np.percentile(bootstrap_accs, 97.5)

# Subgroup analysis
acc_subgroup_0 = accuracy_score(y_test[subgroup == 0], y_pred[subgroup == 0])
acc_subgroup_1 = accuracy_score(y_test[subgroup == 1], y_pred[subgroup == 1])

print("CLAIM AUDIT: 'Our model achieves 95% accuracy'")
print("=" * 70)

print(f"\n1. WHAT IS THE TEST SET?")
print(f"   - Test set size: {len(y_test)}")
print(f"   - Data source: make_moons synthetic data")
print(f"   - Class distribution: {(y_test==0).sum()} class 0, {(y_test==1).sum()} class 1")

print(f"\n2. WHAT IS THE REPORTED ACCURACY?")
print(f"   - Reported accuracy: ~95% (from claim)")
print(f"   - Actual accuracy: {overall_acc:.2%}")
print(f"   - Match: {'No, claim is inflated!' if overall_acc < 0.95 else 'Yes, claim is supported.'}")

print(f"\n3. WHAT IS THE BASELINE?")
print(f"   - Baseline (always predict majority): {baseline_acc:.2%}")
print(f"   - Model improvement over baseline: {overall_acc - baseline_acc:.2%}")
print(f"   - Assessment: Model is {overall_acc / baseline_acc:.2f}x the baseline")

print(f"\n4. WHAT IS THE CONFIDENCE INTERVAL?")
print(f"   - 95% CI: [{ci_lower:.2%}, {ci_upper:.2%}]")
print(f"   - Width: {ci_upper - ci_lower:.2%}")
print(f"   - Assessment: Accuracy could range by {(ci_upper - ci_lower):.2%} due to sampling")

print(f"\n5. ARE THERE SUBGROUP FAILURES?")
print(f"   - Subgroup 0 accuracy: {acc_subgroup_0:.2%} (n={subgroup.sum() == 0})")
print(f"   - Subgroup 1 accuracy: {acc_subgroup_1:.2%} (n={(subgroup == 1).sum()})")
print(f"   - Disparity: {abs(acc_subgroup_0 - acc_subgroup_1):.2%}")
if abs(acc_subgroup_0 - acc_subgroup_1) > 0.05:
    print(f"   - WARNING: Significant disparity between subgroups!")

print(f"\n" + "=" * 70)
print(f"HONEST CLAIM:")
print(f"Our model achieves {overall_acc:.2%} accuracy on the test set (95% CI: [{ci_lower:.2%}, {ci_upper:.2%}]),")
print(f"compared to a {baseline_acc:.2%} baseline. However, performance on Subgroup 1 ({acc_subgroup_1:.2%})")
print(f"is substantially lower than Subgroup 0 ({acc_subgroup_0:.2%}), suggesting the model may underperform")
print(f"for certain populations. Further investigation is needed.")

In [ ]:
# Visualize claim audit
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. Model vs baseline
ax = axes[0, 0]
models = ['Baseline\n(Majority)', 'Our Model']
accs = [baseline_acc, overall_acc]
colors_bar = ['gray', '#4472C4']
bars = ax.bar(models, accs, color=colors_bar, alpha=0.7, edgecolor='black')
ax.axhline(0.95, color='red', linestyle='--', linewidth=2, label='Claimed: 95%')
ax.set_ylabel('Accuracy')
ax.set_title('Model vs. Baseline')
ax.set_ylim([0, 1.0])
ax.legend()
ax.grid(axis='y', alpha=0.3)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, acc + 0.02, f'{acc:.2%}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

# 2. Confidence interval
ax = axes[0, 1]
ax.barh([0], [ci_upper - ci_lower], left=ci_lower, height=0.3, color='steelblue', alpha=0.7)
ax.plot([overall_acc], [0], 'ro', markersize=10, label='Point estimate')
ax.set_xlim([0.70, 1.0])
ax.set_yticks([])
ax.set_xlabel('Accuracy')
ax.set_title('95% Bootstrap Confidence Interval')
ax.axvline(0.95, color='red', linestyle='--', linewidth=2, alpha=0.5)
ax.text(ci_lower, 0.25, f'{ci_lower:.3f}', ha='center', fontsize=10)
ax.text(ci_upper, 0.25, f'{ci_upper:.3f}', ha='center', fontsize=10)
ax.legend()
ax.grid(axis='x', alpha=0.3)

# 3. Distribution of bootstrap accuracies
ax = axes[1, 0]
ax.hist(bootstrap_accs, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
ax.axvline(overall_acc, color='red', linestyle='--', linewidth=2, label='Point estimate')
ax.axvline(ci_lower, color='orange', linestyle=':', linewidth=2)
ax.axvline(ci_upper, color='orange', linestyle=':', linewidth=2, label='95% CI bounds')
ax.set_xlabel('Accuracy')
ax.set_ylabel('Frequency')
ax.set_title('Bootstrap Distribution of Test Accuracy')
ax.legend()
ax.grid(alpha=0.3)

# 4. Subgroup performance
ax = axes[1, 1]
subgroups = ['Subgroup 0', 'Subgroup 1']
accs_sub = [acc_subgroup_0, acc_subgroup_1]
colors_sub = ['#2ca02c', '#d62728']
bars = ax.bar(subgroups, accs_sub, color=colors_sub, alpha=0.7, edgecolor='black')
ax.axhline(overall_acc, color='blue', linestyle='--', linewidth=2, label='Overall accuracy')
ax.set_ylabel('Accuracy')
ax.set_title('Subgroup Performance Disparity')
ax.set_ylim([0, 1.0])
ax.legend()
ax.grid(axis='y', alpha=0.3)
for bar, acc in zip(bars, accs_sub):
    ax.text(bar.get_x() + bar.get_width()/2, acc + 0.02, f'{acc:.2%}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.suptitle('Claim Audit: Critical Analysis of "95% Accuracy" Claim', 
             fontsize=13, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## What to Notice

1. **Repeated Runs Are Essential**: A single run can be misleading. Accuracy varies significantly due to random train/test splits and model initialization. Always report mean ± std over multiple runs.

2. **Ablation Studies Reveal Feature Value**: Only useful features cause a significant drop when removed. Noise features contribute nothing. This guides feature engineering decisions.

3. **Slice Analysis Uncovers Failure Modes**: Overall metrics hide subgroup disparities. Always inspect performance on meaningful slices (demographics, regions, conditions).

4. **Simpson's Paradox Is Real**: Aggregate metrics can reverse when disaggregated. This happens when the composition of the subgroups differs. Always report both aggregate and slice metrics.

5. **Claims Must Be Auditable**: "95% accuracy" is vague. Honest claims include: test set definition, comparison to baseline, confidence intervals, and disclosure of subgroup failures.

6. **Error Analysis Is Not Optional**: Understanding *where* your model fails is as important as knowing *that* it fails. Use confusion matrices, subgroup analysis, and slice analysis to guide improvements.